# Variantes de denoising con cinco semillas, Mirai

Compara con el control las configuraciones que lo superan con la semilla 42, además del masking mediante Dropout al 10 %, que es la variante aplicada al tráfico real. Se repite con las semillas 42, 1, 2, 3 y 4. La partición de los datos es fija, de modo que solo varía el azar del entrenamiento. La dirección IP de origen (`src_ip`) no se usa como entrada, porque identifica a los equipos atacantes.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path("../../data/flows.csv")

df = pd.read_csv(DATA_PATH).drop(columns=['class_name', 'flow_start'])
print(df.shape)

(1006776, 11)


In [3]:
from sklearn.model_selection import train_test_split

y = df['label'].values
X = pd.get_dummies(df.drop(columns=['label', 'src_ip']), columns=['state'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (805420, 15), X_train_normal: (6234, 15), X_test: (201356, 15)


In [4]:
class PureMasking(layers.Layer):
    """Anula una fracción mask_rate de las columnas sin reescalar el resto, a diferencia de Dropout."""
    def __init__(self, mask_rate, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate

    def call(self, inputs, training=None):
        if training:
            keep_mask = tf.cast(tf.random.uniform(tf.shape(inputs)) >= self.mask_rate, inputs.dtype)
            return inputs * keep_mask
        return inputs


CORRUPTION = {
    'gaussiano': layers.GaussianNoise,
    'dropout': layers.Dropout,
    'puro': PureMasking,
}


def build_dae(input_dim, hidden_units, kind=None, level=0.0):
    """Autoencoder denso con una capa de corrupción opcional a la entrada (kind=None es el control)."""
    inputs = keras.Input(shape=(input_dim,))
    x = CORRUPTION[kind](level)(inputs) if kind else inputs
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)

In [5]:
HIDDEN_UNITS = [64, 32, 16]
CONFIGS = [('control', 0.0)] + [('dropout', 0.1), ('dropout', 0.2), ('puro', 0.1)]
SEEDS = [42, 1, 2, 3, 4]

results = {}
for seed in SEEDS:
    for kind, level in CONFIGS:
        tf.keras.utils.set_random_seed(seed)
        model = build_dae(X_train.shape[1], HIDDEN_UNITS, None if kind == 'control' else kind, level)
        history = model.fit(
            X_train_normal, X_train_normal,
            epochs=100,
            batch_size=256,
            validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
            verbose=0,
        )
        auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
        results[(f"{kind} {level}", seed)] = auc
        print(f"seed={seed} {kind}={level}: epocas={len(history.history['loss'])}, ROC-AUC={auc:.4f}", flush=True)

seed=42 control=0.0: epocas=100, ROC-AUC=0.9442


seed=42 dropout=0.1: epocas=75, ROC-AUC=0.9542


seed=42 dropout=0.2: epocas=53, ROC-AUC=0.9569


seed=42 puro=0.1: epocas=70, ROC-AUC=0.9672


seed=1 control=0.0: epocas=100, ROC-AUC=0.9356


...

seed=3 puro=0.1: epocas=60, ROC-AUC=0.8955


seed=4 control=0.0: epocas=100, ROC-AUC=0.9254


seed=4 dropout=0.1: epocas=79, ROC-AUC=0.8471


seed=4 dropout=0.2: epocas=84, ROC-AUC=0.6797


seed=4 puro=0.1: epocas=100, ROC-AUC=0.8887


In [6]:
from scipy.stats import ttest_rel

auc = pd.Series(results).unstack()  # filas: configuración, columnas: semilla
print("AUC por configuracion y semilla:")
print(auc.round(4).to_string())

control = auc.loc['control 0.0']
new = [s for s in SEEDS if s != 42]
print("\nResumen frente al control de la misma semilla:")
for name, row in auc.iterrows():
    line = f"  {name:14s} media={row.mean():.4f} std={row.std(ddof=1):.4f} media 4 nuevas={row[new].mean():.4f}"
    if name != 'control 0.0':
        d = row - control
        line += (f" | dif={d.mean():+.4f} mejora en {int((d > 0).sum())}/5 ({int((d[new] > 0).sum())}/4 nuevas)"
                 f" p={ttest_rel(row, control).pvalue:.3f}")
    print(line)

AUC por configuracion y semilla:
                 1       2       3       4       42
control 0.0  0.9356  0.9409  0.9404  0.9254  0.9442
dropout 0.1  0.7189  0.7658  0.8444  0.8471  0.9542
dropout 0.2  0.6198  0.6447  0.8515  0.6797  0.9569
puro 0.1     0.8919  0.8351  0.8955  0.8887  0.9672

Resumen frente al control de la misma semilla:
  control 0.0    media=0.9373 std=0.0073 media 4 nuevas=0.9356
  dropout 0.1    media=0.8261 std=0.0899 media 4 nuevas=0.7940 | dif=-0.1112 mejora en 1/5 (0/4 nuevas) p=0.048
  dropout 0.2    media=0.7505 std=0.1467 media 4 nuevas=0.6989 | dif=-0.1868 mejora en 1/5 (0/4 nuevas) p=0.043
  puro 0.1       media=0.8957 std=0.0470 media 4 nuevas=0.8778 | dif=-0.0416 mejora en 1/5 (0/4 nuevas) p=0.111
